# Resume Screener: run the actual LoRA adapter

This notebook loads **Qwen2.5-0.5B-Instruct + the trained LoRA adapter** ([Kus-hal/resume-screener-lora](https://huggingface.co/Kus-hal/resume-screener-lora)) and screens a resume for a target role. It returns the structured JSON the adapter was trained to emit.

Use **Runtime > Run all**. A free GPU runtime (**Runtime > Change runtime type > T4 GPU**) is faster, but CPU works too. The first run downloads about 1 GB of model files.

Code, evaluation and limitations: [github.com/ayush-s-tomar/resume-screener-lora](https://github.com/ayush-s-tomar/resume-screener-lora). The labels behind this adapter are synthetic, so treat the scores as a demo of the model's behaviour, not as hiring advice.

In [ ]:
!pip install -q peft accelerate

In [ ]:
import json
import re

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE = "Qwen/Qwen2.5-0.5B-Instruct"
ADAPTER = "Kus-hal/resume-screener-lora"
device = "cuda" if torch.cuda.is_available() else "cpu"

tok = AutoTokenizer.from_pretrained(BASE)
base_model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float32)
model = PeftModel.from_pretrained(base_model, ADAPTER).to(device).eval()
print("Loaded the adapter on", device)

In [ ]:
def screen(role, resume, max_new_tokens=200):
    prompt = f"Screen this resume for a {role} position and return a structured verdict.\n\nResume: {resume}"
    inputs = tok.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    text = tok.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return {"raw_output": text}
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return {"raw_output": text}

## Try it

Edit `role` and `resume`, then run the cell. Below it, a resume from a different field shows how the adapter handles a mismatch.

In [ ]:
role = "Backend Engineer"
resume = '''Backend Engineer, 5 years of experience.
Skills: Python, FastAPI, PostgreSQL, Redis, Docker, Kubernetes, REST APIs, AWS.
Experience: Built and scaled payment microservices handling 2M requests/day. Designed REST APIs with FastAPI and PostgreSQL, cut p95 latency by 40% with Redis caching, and ran services on Kubernetes with CI/CD.
Education: B.Tech in Computer Science.'''

print(json.dumps(screen(role, resume), indent=2))

In [ ]:
mismatch = '''Graphic Designer, 6 years of experience.
Skills: Photoshop, Illustrator, Figma, Brand Identity, Typography.
Experience: Designed branding and print campaigns for retail clients. Led a team of 3 designers on packaging projects.
Education: Bachelor of Fine Arts.'''

print(json.dumps(screen("Backend Engineer", mismatch), indent=2))

## Optional: a small web UI

This starts a temporary public link (valid while the notebook runs) with the same model behind a form.

In [ ]:
!pip install -q gradio
import gradio as gr

ui = gr.Interface(
    fn=screen,
    inputs=[gr.Textbox(value="Backend Engineer", label="Target role"), gr.Textbox(lines=10, label="Resume text")],
    outputs=gr.JSON(label="Model output (JSON)"),
    title="Resume Screener: LoRA adapter",
)
ui.launch(share=True)